In [ ]:
import pandas as pd

# 1. Definir o nome do novo arquivo
arquivo_saida = "06_PENSE2024_SEM_RESPOSTA_SEM_ABANDONO.parquet"

# 2. Identificar apenas as colunas de perguntas (ignora colunas de IDs/pesos)
colunas_perguntas = [col for col in df.columns if col.startswith("B") or col == "IDADE_AGREG"]

# 3. Criar a máscara de exclusão de forma eficiente
mascara_excluir = df[colunas_perguntas].astype(str).isin(["-9", "-2"]).any(axis=1)

# 4. Contabilizar registros
total_original = len(df)
total_excluir = mascara_excluir.sum()

# 5. Criar novo DataFrame filtrado
df_limpo = df.loc[~mascara_excluir].copy()

# 6. Salvar em Parquet
df_limpo.to_parquet(arquivo_saida, index=False)

# Resumo no terminal
print("========== PROCESSAMENTO CONCLUÍDO ==========")
print(f"Linhas originais: {total_original:,}")
print(f"Linhas removidas: {total_excluir:,}")
print(f"Linhas restantes: {len(df_limpo):,}")
print(f"Percentual removido: {total_excluir / total_original * 100:.2f}%")

========== PROCESSAMENTO CONCLUÍDO ==========
Linhas originais: 148,707
Linhas removidas: 5,778
Linhas restantes: 142,929
Percentual removido: 3.89%


In [ ]:
import pandas as pd
from IPython.display import display, Markdown

# 1. Carregar o dicionário de dados
dicionario = pd.read_excel("00_Dicionário_Microdados_PENSE2024.xlsx")

dico_map = {}
for _, row in dicionario.iterrows():
    desc = str(row["Descrição da Variável"]).strip()
    cat = str(row["Categoria"]) if pd.notna(row["Categoria"]) else ""
    cat_dict = {}
    if cat:
        for linha in cat.split("\n"):
            if " - " in linha:
                partes = linha.split(" - ", 1)
                cat_dict[partes[0].strip()] = partes[1].strip()
    dico_map[desc] = cat_dict

# 2. Gerar a tabela em Markdown
total = len(df_limpo)
markdown_lines = [
    "| Pergunta / Variável | Distribuição das Respostas |",
    "| :--- | :--- |"
]

for col in df_limpo.columns:
    counts = df_limpo[col].astype(str).str.strip().value_counts(dropna=False)
    cat_dict = dico_map.get(col, {})
    
    respostas_formatadas = []
    for cod, qtd in counts.items():
        rotulo = cat_dict.get(cod, "")
        pct = (qtd / total) * 100
        if rotulo:
            respostas_formatadas.append(f"[{cod}] {rotulo}: {pct:.2f}%")
        else:
            respostas_formatadas.append(f"[{cod}]: {pct:.2f}%")
            
    # Subsitui quebras de linha normais por <br> do HTML para a tabela Markdown respeitar a formatação
    distribuicao_str = "<br>".join(respostas_formatadas)
    
    # Limpa quebras de linha no nome da pergunta para não quebrar a tabela Markdown
    col_clean = str(col).replace("\n", " ").replace("|", "-")
    
    markdown_lines.append(f"| {col_clean} | {distribuicao_str} |")

# 3. Exibir a tabela renderizada
print("========== EXIBIÇÃO DOS RESULTADOS NO IPYNB ==========\n")
display(Markdown("\n".join(markdown_lines)))